# 1. Creacion estructura medallion: Gold y Silver

In [0]:
-- Ejercicio 1.1
-- verificacion de catalogo
SHOW SCHEMAS IN bootcamp_de_valentin;

SHOW TABLES IN bootcamp_de_valentin.bronze;

SELECT COUNT(*) FROM bootcamp_de_valentin.bronze.propiedades_bronze;

In [0]:
-- Ejercicio 1.2
-- Crear schemas Silver y Gold
CREATE SCHEMA IF NOT EXISTS bootcamp_de_valentin.silver
COMMENT 'Schema silver para almacenar datos proceados, transformados y limpios. Bootcamp DE - Luciano Argolo';

CREATE SCHEMA IF NOT EXISTS bootcamp_de_valentin.gold
COMMENT 'Schema gold para almacenar datos modelados para el analisis de negocio con metricas y agregaciones necesarias. Bootcamp DE - Luciano Argolo';

In [0]:
SHOW SCHEMAS IN bootcamp_de_valentin

# 2. Crear Tabla Silver

**Ejercicio 2.1: Diseñar Tabla Silver**

Columnas necesarias basadas en el EDA de Semana 2:

- Ubicación: partido, region (derivadas de zona en Bronze)
- Precio: tipo_operacion, precio, moneda, expensas, precio_por_m2
- Características: ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, antiguedad, cochera, orientacion, estado, url, fecha_publicacion
- Metadata: _source_table, _processing_timestamp

In [0]:
-- Ejercicio 2.2
-- Crear Tabla Silver
DROP TABLE IF EXISTS bootcamp_de_valentin.silver.propiedades_silver;
CREATE TABLE bootcamp_de_valentin.silver.propiedades_silver
(
  propiedad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Id unico de la propiedad (auto-generado)',

  partido STRING COMMENT 'Partido/Municipio estandarizado',
  region STRING COMMENT 'Region geografica: capital federal, gba zona norte/oeste/sur',

  tipo_operacion STRING COMMENT 'Alquiler, venta, alquiler_temporario',
  precio DECIMAL(15,2) COMMENT 'Precio de la propiedad',
  moneda STRING COMMENT 'USD o ARS',
  expensas DECIMAL(15,2) COMMENT 'Expensas mensuales',
  precio_por_m2 DECIMAL(15,2) COMMENT 'Precio por metro cuadrado',

  ambientes INT COMMENT 'Cantidad de ambientes',
  m2_totales DECIMAL(15,2) COMMENT 'Superficie total',
  m2_cubiertos DECIMAL(15,2) COMMENT 'Superficie cubierta',
  antiguedad INT COMMENT 'Años de antiguedad (999 si no disponible)',
  cochera BOOLEAN COMMENT 'Tiene cochera o no',
  orientacion STRING COMMENT 'Orientacion del inmueble',
  estado STRING COMMENT 'Estado de la propiedad',
  url STRING COMMENT 'URL de la propiedad',
  fecha_publicacion DATE COMMENT 'Fecha de publicacion - desde fecha (STRING) en Bronze',

  _source_table STRING DEFAULT 'bootcamp_de_valentin.bronze.propiedades_bronze' COMMENT 'Tabla de origen',
  _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP() COMMENT 'Timestamp de procesamiento'
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Propiedades inmobiliarias - Capa Silver (datos limpios y validados)';

In [0]:
-- Ejercicio 2.3
-- Verificacion de estructura creada
DESCRIBE EXTENDED bootcamp_de_valentin.silver.propiedades_silver

# 3. Transformacion Bronze --> Silver

In [0]:
-- Ejercicio 3.1 
-- Ejecutar limpieza de Semana 2
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean AS
SELECT
  CASE WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double ELSE NULL END AS precio,
  moneda,
  CASE WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double ELSE NULL END AS ambientes,
  CASE WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double ELSE NULL END AS m2_totales,
  CASE WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double ELSE NULL END AS m2_cubiertos,
  CASE WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double ELSE NULL END AS antiguedad,
  tipo_de_operacion,
  id,
  ubicacion,
  numero,
  calle,
  expensas,
  orientacion_cardinal,
  orientacion_inmueble,
  piso,
  cochera,
  estado,
  tipo_vendedor,
  url,
  zona,
  fecha,
  hora
FROM bootcamp_de_valentin.bronze.propiedades_bronze;

-- propiedades_clean_2: filtro de outliers por percentiles (Semana 2)
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
  WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE 
      precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT 
    p.*
  FROM propiedades_clean p
  JOIN limites l
    ON p.moneda = l.moneda
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE 
    p.precio BETWEEN l.p01 AND l.p99
);

-- bronze_EDA: limpieza detallada (Semana 2)
CREATE OR REPLACE TEMP VIEW bronze_EDA AS (
  SELECT
    edl.id,
    edl.ubicacion,
    CASE
      WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.precio::float BETWEEN 0 AND 2147483648 THEN edl.precio::float
      ELSE NULL
    END AS precio,
    CASE
      WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
      ELSE NULL
    END AS numero,
    edl.calle,
    CASE
      WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      WHEN edl.expensas::float BETWEEN 0 AND 20000000 THEN edl.expensas::float
      ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
      WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
      WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
      WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
      WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
      ELSE edl.moneda
    END AS moneda,
    CASE
      WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.ambientes::float
    END AS ambientes,
    CASE
      WHEN edl.m2_totales = 'NaN' OR edl.m2_totales NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.m2_totales::decimal
    END AS m2_totales,
    CASE
      WHEN edl.m2_cubiertos = 'NaN' OR edl.m2_cubiertos NOT RLIKE '^-?[0-9]+\\.?[0-9]*$' THEN NULL
      ELSE edl.m2_cubiertos::decimal
    END AS m2_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
      WHEN edl.piso IS NULL THEN NULL
      WHEN edl.piso = 'NaN' THEN NULL
      ELSE edl.piso::float
    END AS piso,
    CASE
      WHEN edl.cochera = 'tiene' THEN 1
      ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    COALESCE(CAST(edl.fecha AS DATE), CURRENT_DATE()) AS fecha
  FROM propiedades_clean_2 edl
  WHERE 
    edl.url RLIKE 'https'
    AND (edl.zona LIKE '%gba%' OR edl.zona LIKE '%caba%' OR edl.zona LIKE '%capital%')
    AND edl.zona LIKE '%-%'
    AND LENGTH(edl.zona) < 50
);

SELECT COUNT(*) as registros_bronze_eda FROM bronze_EDA;

In [0]:
-- Ejercicio 3.2.1
-- Pipeline Bronze_EDA --> Silver, Datos Tipados
WITH datos_tipados AS(
  SELECT
    LOWER(TRIM(zona)) AS zona,
    LOWER(TRIM(tipo_de_operacion)) AS tipo_operacion,
    CAST(precio AS DECIMAL(15,2)) AS precio,
    moneda,
    COALESCE(CAST(expensas AS DECIMAL(15,2)),0) AS expensas,
    CAST(ambientes as INT) AS ambientes,
    CAST(m2_totales AS DECIMAL(15,2)) AS m2_totales,
    CAST(m2_cubiertos AS DECIMAL(15,2)) AS m2_cubiertos,
    CASE
      WHEN antiguedad RLIKE '^[0-9]' AND CAST (antiguedad AS FLOAT) BETWEEN 0 AND 200 THEN CAST(antiguedad as INT)
      ELSE 999
    END AS antiguedad,
    CASE
      WHEN cochera = 1 THEN TRUE
      ELSE FALSE
    END AS cochera,
    COALESCE(TRIM(orientacion_inmueble),'-1') AS orientacion,
    COALESCE(TRIM(estado),'-1') AS ESTADO,
    URL,
    fecha as fecha_publicacion
  FROM bronze_EDA
  WHERE
    precio > 0
    AND m2_totales > 0 
    AND tipo_de_operacion IS NOT NULL
)
SELECT 
  *,
  typeof(precio),
  typeof(ambientes) 
FROM datos_tipados
LIMIT 10

In [0]:
-- Ejercicio 3.2.2
-- Datos Deduplicados
WITH datos_tipados AS(
  SELECT
    LOWER(TRIM(zona)) AS zona,
    LOWER(TRIM(tipo_de_operacion)) AS tipo_operacion,
    CAST(precio AS DECIMAL(15,2)) AS precio,
    moneda,
    COALESCE(CAST(expensas AS DECIMAL(15,2)),0) AS expensas,
    CAST(ambientes as INT) AS ambientes,
    CAST(m2_totales AS DECIMAL(15,2)) AS m2_totales,
    CAST(m2_cubiertos AS DECIMAL(15,2)) AS m2_cubiertos,
    CASE
      WHEN antiguedad RLIKE '^[0-9]' AND CAST (antiguedad AS FLOAT) BETWEEN 0 AND 200 THEN CAST(antiguedad as INT)
      ELSE 999
    END AS antiguedad,
    CASE
      WHEN cochera = 1 THEN TRUE
      ELSE FALSE
    END AS cochera,
    COALESCE(TRIM(orientacion_inmueble),'-1') AS orientacion,
    COALESCE(TRIM(estado),'-1') AS ESTADO,
    URL,
    fecha as fecha_publicacion
  FROM bronze_EDA
  WHERE
    precio > 0
    AND m2_totales > 0 
    AND tipo_de_operacion IS NOT NULL
),
datos_deduplicados AS (
  SELECT
    *,
    row_number() OVER (PARTITION BY precio, url ORDER BY precio) rn
  FROM datos_tipados
)
-- SELECT
--   *
-- FROM datos_deduplicados
-- -- WHERE rn > 1
-- ORDER BY precio, url, rn
SELECT
  COUNT(*),
  COUNT(CASE WHEN rn = 1 THEN 1 END) AS unicos,
  COUNT(CASE WHEN rn > 1 THEN 1 END) AS duplicados
FROM datos_deduplicados

In [0]:
-- Ejercicio 3.2.3
-- Agregar mapeo de partido y region, precio por m2, valores deduplicados y tipados
WITH datos_tipados AS(
  SELECT
    LOWER(TRIM(zona)) AS zona,
    LOWER(TRIM(tipo_de_operacion)) AS tipo_operacion,
    CAST(precio AS DECIMAL(15,2)) AS precio,
    moneda,
    COALESCE(CAST(expensas AS DECIMAL(15,2)),0) AS expensas,
    CAST(ambientes as INT) AS ambientes,
    CAST(m2_totales AS DECIMAL(15,2)) AS m2_totales,
    CAST(m2_cubiertos AS DECIMAL(15,2)) AS m2_cubiertos,
    CASE
      WHEN antiguedad RLIKE '^[0-9]' AND CAST (antiguedad AS FLOAT) BETWEEN 0 AND 200 THEN CAST(antiguedad as INT)
      ELSE 999
    END AS antiguedad,
    CASE
      WHEN cochera = 1 THEN TRUE
      ELSE FALSE
    END AS cochera,
    COALESCE(TRIM(orientacion_inmueble),'-1') AS orientacion,
    COALESCE(TRIM(estado),'-1') AS ESTADO,
    URL,
    fecha as fecha_publicacion
  FROM bronze_EDA
  WHERE
    precio > 0
    AND m2_totales > 0 
    AND tipo_de_operacion IS NOT NULL
),
datos_deduplicados AS (
  SELECT
    *,
    row_number() OVER (PARTITION BY precio, url ORDER BY precio) rn
  FROM datos_tipados
),
datos_finales(
  SELECT
    CASE
      -- Capital Federal
      WHEN zona = 'capital-federal' THEN 'capital federal'
      -- GBA Norte
      WHEN zona = 'bsas-gba-norte/vicente-lopez' THEN 'vicente lopez'
      WHEN zona = 'bsas-gba-norte/pilar' THEN 'pilar'
      WHEN zona = 'bsas-gba-norte/tigre' THEN 'tigre'
      WHEN zona = 'bsas-gba-norte/san-isidro' THEN 'san isidro'
      WHEN zona = 'bsas-gba-norte/general-san-martin' THEN 'general san martin'
      WHEN zona = 'bsas-gba-norte/san-fernando' THEN 'san fernando'
      WHEN zona = 'bsas-gba-norte/san-miguel' THEN 'san miguel'
      WHEN zona = 'bsas-gba-norte/malvinas-argentinas' THEN 'malvinas argentinas'
      WHEN zona = 'bsas-gba-norte/escobar' THEN 'escobar'
      -- GBA Oeste
      WHEN zona = 'bsas-gba-oeste/caseros' THEN 'tres de febrero'
      WHEN zona = 'bsas-gba-oeste/castelar' THEN 'moron'
      WHEN zona = 'bsas-gba-oeste/general-rodriguez' THEN 'general rodriguez'
      WHEN zona = 'bsas-gba-oeste/hurlingham' THEN 'hurlingham'
      WHEN zona = 'bsas-gba-oeste/ituzaingo' THEN 'ituzaingo'
      WHEN zona = 'bsas-gba-oeste/la-matanza' THEN 'la matanza'
      WHEN zona = 'bsas-gba-oeste/merlo' THEN 'merlo'
      WHEN zona = 'bsas-gba-oeste/moreno' THEN 'moreno'
      WHEN zona = 'bsas-gba-oeste/moron' THEN 'moron'
      WHEN zona = 'bsas-gba-oeste/tres-de-febrero' THEN 'tres de febrero'
      -- GBA Sur
      WHEN zona = 'bsas-gba-sur/almirante-brown' THEN 'almirante brown'
      WHEN zona = 'bsas-gba-sur/avellaneda' THEN 'avellaneda'
      WHEN zona = 'bsas-gba-sur/berazategui' THEN 'berazategui'
      WHEN zona = 'bsas-gba-sur/esteban-echeverria' THEN 'esteban echeverria'
      WHEN zona = 'bsas-gba-sur/ezeiza' THEN 'ezeiza'
      WHEN zona = 'bsas-gba-sur/florencio-varela' THEN 'florencio varela'
      WHEN zona = 'bsas-gba-sur/lanus' THEN 'lanus'
      WHEN zona = 'bsas-gba-sur/la-plata' THEN 'la plata'
      WHEN zona = 'bsas-gba-sur/lomas-de-zamora' THEN 'lomas de zamora'
      WHEN zona = 'bsas-gba-sur/quilmes' THEN 'quilmes'
      -- Argenprop specific mappings
      WHEN zona RLIKE 'vicente-lopez' THEN 'vicente lopez'
      WHEN zona RLIKE 'pilar' THEN 'pilar'
      WHEN zona RLIKE 'general-san-martin' THEN 'general san martin'
      WHEN zona RLIKE 'tigre' THEN 'tigre'
      WHEN zona RLIKE 'san-fernando' THEN 'san fernando'
      WHEN zona RLIKE 'san-miguel' THEN 'san miguel'
      WHEN zona RLIKE 'malvinas-argentinas' THEN 'malvinas argentinas'
      WHEN zona RLIKE 'escobar' THEN 'escobar'
      WHEN zona RLIKE 'caseros' THEN 'tres de febrero'
      WHEN zona RLIKE 'castelar' THEN 'moron'
      WHEN zona RLIKE 'general-rodriguez' THEN 'general rodriguez'
      WHEN zona RLIKE 'hurlingham' THEN 'hurlingham'
      WHEN zona RLIKE 'ituzaingo' THEN 'ituzaingo'
      WHEN zona RLIKE 'la-matanza' THEN 'la matanza'
      WHEN zona RLIKE 'merlo' THEN 'merlo'
      WHEN zona RLIKE 'moreno' THEN 'moreno'
      WHEN zona RLIKE 'moron' THEN 'moron'
      WHEN zona RLIKE 'tres-de-febrero' THEN 'tres de febrero'
      WHEN zona RLIKE 'almirante-brown' THEN 'almirante brown'
      WHEN zona RLIKE 'avellaneda' THEN 'avellaneda'
      WHEN zona RLIKE 'berazategui' THEN 'berazategui'
      WHEN zona RLIKE 'berisso' THEN 'berisso'
      WHEN zona RLIKE 'ensenada' THEN 'ensenada'
      WHEN zona RLIKE 'esteban-echeverria' THEN 'esteban echeverria'
      WHEN zona RLIKE 'ezeiza' THEN 'ezeiza'
      WHEN zona RLIKE 'florencio-varela' THEN 'florencio varela'
      WHEN zona RLIKE 'lanus' THEN 'lanus'
      WHEN zona RLIKE 'la-plata' THEN 'la plata'
      WHEN zona RLIKE 'lomas-de-zamora' THEN 'lomas de zamora'
      WHEN zona RLIKE 'presidente-peron' THEN 'presidente peron'
      WHEN zona RLIKE 'quilmes' THEN 'quilmes'
      WHEN zona RLIKE 'san-vicente' THEN 'san vicente'
      -- ZonaProp specific mappings
      WHEN zona = 'san-isidro' THEN 'san isidro'
      -- LiderProp specific mappings
      WHEN zona = 'gba-zona-norte--vicente-lopez' THEN 'vicente lopez'
      WHEN zona = 'gba-zona-norte--pilar' THEN 'pilar'
      WHEN zona = 'gba-zona-norte--general-san-martin' THEN 'general san martin'
      WHEN zona = 'gba-zona-norte--san-isidro' THEN 'san isidro'
      WHEN zona = 'gba-zona-norte--tigre' THEN 'tigre'
      WHEN zona = 'gba-zona-norte--san-fernando' THEN 'san fernando'
      WHEN zona = 'gba-zona-norte--san-miguel' THEN 'san miguel'
      WHEN zona = 'gba-zona-norte--malvinas-argentinas' THEN 'malvinas argentinas'
      WHEN zona = 'gba-zona-norte--escobar' THEN 'escobar'
      WHEN zona = 'gba-zona-oeste--canuelas' THEN 'canuelas'
      WHEN zona = 'gba-zona-oeste--general-rodriguez' THEN 'general rodriguez'
      WHEN zona = 'gba-zona-oeste--hurlingham' THEN 'hurlingham'
      WHEN zona = 'gba-zona-oeste--ituzaingo' THEN 'ituzaingo'
      WHEN zona = 'gba-zona-oeste--la-matanza' THEN 'la matanza'
      WHEN zona = 'gba-zona-oeste--lujan' THEN 'lujan'
      WHEN zona = 'gba-zona-oeste--marcos-paz' THEN 'marcos paz'
      WHEN zona = 'gba-zona-oeste--merlo' THEN 'merlo'
      WHEN zona = 'gba-zona-oeste--moreno' THEN 'moreno'
      WHEN zona = 'gba-zona-oeste--moron' THEN 'moron'
      WHEN zona = 'gba-zona-oeste--tres-de-febrero' THEN 'tres de febrero'
      WHEN zona = 'gba-zona-sur--almirante-brown' THEN 'almirante brown'
      WHEN zona = 'gba-zona-sur--avellaneda' THEN 'avellaneda'
      WHEN zona = 'gba-zona-sur--berazategui' THEN 'berazategui'
      WHEN zona = 'gba-zona-sur--ensenada' THEN 'ensenada'
      WHEN zona = 'gba-zona-sur--san-vicente' THEN 'san vicente'
      WHEN zona = 'gba-zona-sur--esteban-echeverria' THEN 'esteban echeverria'
      WHEN zona = 'gba-zona-sur--ezeiza' THEN 'ezeiza'
      WHEN zona = 'gba-zona-sur--florencio-varela' THEN 'florencio varela'
      WHEN zona = 'gba-zona-sur--lanus' THEN 'lanus'
      WHEN zona = 'gba-zona-sur--la-plata' THEN 'la plata'
      WHEN zona = 'gba-zona-sur--lomas-de-zamora' THEN 'lomas de zamora'
      WHEN zona = 'gba-zona-sur--presidente-peron' THEN 'presidente peron'
      WHEN zona = 'gba-zona-sur--quilmes' THEN 'quilmes'
      ELSE 'no especifica'
    END AS partido,
    CASE
      -- Capital Federal
      WHEN zona IS NULL THEN 'no especifica'
      WHEN zona = 'capital-federal' THEN 'capital federal'
      -- GBA Norte
      WHEN zona LIKE 'bsas-gba-norte/%' THEN 'gba zona norte'
      WHEN zona IN ('vicente-lopez', 'pilar', 'tigre', 'san-isidro', 'general-san-martin',
                    'san-fernando', 'san-miguel', 'jose-c-paz', 'malvinas-argentinas', 'escobar') THEN 'gba zona norte'
      -- GBA Oeste
      WHEN zona LIKE 'bsas-gba-oeste/%' THEN 'gba zona oeste'
      WHEN zona IN ('caseros', 'castelar', 'general-rodriguez', 'hurlingham', 'ituzaingo',
                    'la-matanza', 'merlo', 'moreno', 'moron', 'tres-de-febrero') THEN 'gba zona oeste'
      -- GBA Sur
      WHEN zona LIKE 'bsas-gba-sur/%' THEN 'gba zona sur'
      WHEN zona IN ('almirante-brown', 'avellaneda', 'berazategui', 'berisso', 'ensenada',
                    'esteban-echeverria', 'ezeiza', 'florencio-varela', 'lanus', 'la-plata',
                    'lomas-de-zamora', 'presidente-peron', 'quilmes', 'san-vicente') THEN 'gba zona sur'
      -- LiderProp specific mappings
      WHEN zona LIKE 'gba-zona-norte--%' THEN 'gba zona norte'
      WHEN zona LIKE 'gba-zona-oeste--%' THEN 'gba zona oeste'
      WHEN zona LIKE 'gba-zona-sur--%' THEN 'gba zona sur'
      ELSE zona
    END AS region,
    tipo_operacion,
    precio,
    moneda,
    expensas,
    ambientes,
    m2_totales,
    m2_cubiertos,
    antiguedad,
    cochera,
    orientacion,
    estado,
    url,
    fecha_publicacion,
    ROUND(precio / m2_totales,2) AS precio_por_m2 
  FROM datos_deduplicados
  WHERE
    rn = 1
)
SELECT
  *
FROm datos_finales;

In [0]:
-- Ejercicio 3.2.4
-- INSERT INTO Silver con datos_finales, sumando campos de metada
INSERT INTO bootcamp_de_valentin.silver.propiedades_silver(
  partido,
  region,
  tipo_operacion,
  precio,
  moneda,
  expensas,
  ambientes,
  m2_totales,
  m2_cubiertos,
  antiguedad,
  cochera,
  orientacion,
  estado,
  url,
  fecha_publicacion,
  precio_por_m2,
  _source_table,
  _processing_timestamp
)
WITH datos_tipados AS(
  SELECT
    LOWER(TRIM(zona)) AS zona,
    LOWER(TRIM(tipo_de_operacion)) AS tipo_operacion,
    CAST(precio AS DECIMAL(15,2)) AS precio,
    moneda,
    COALESCE(CAST(expensas AS DECIMAL(15,2)),0) AS expensas,
    CAST(ambientes as INT) AS ambientes,
    CAST(m2_totales AS DECIMAL(15,2)) AS m2_totales,
    CAST(m2_cubiertos AS DECIMAL(15,2)) AS m2_cubiertos,
    CASE
      WHEN antiguedad RLIKE '^[0-9]' AND CAST (antiguedad AS FLOAT) BETWEEN 0 AND 200 THEN CAST(antiguedad as INT)
      ELSE 999
    END AS antiguedad,
    CASE
      WHEN cochera = 1 THEN TRUE
      ELSE FALSE
    END AS cochera,
    COALESCE(TRIM(orientacion_inmueble),'-1') AS orientacion,
    COALESCE(TRIM(estado),'-1') AS ESTADO,
    URL,
    fecha as fecha_publicacion
  FROM bronze_EDA
  WHERE
    precio > 0
    AND m2_totales > 0 
    AND tipo_de_operacion IS NOT NULL
),
datos_deduplicados AS (
  SELECT
    *,
    row_number() OVER (PARTITION BY precio, url ORDER BY precio) rn
  FROM datos_tipados
),
datos_finales(
  SELECT
    CASE
      -- Capital Federal
      WHEN zona = 'capital-federal' THEN 'capital federal'
      -- GBA Norte
      WHEN zona = 'bsas-gba-norte/vicente-lopez' THEN 'vicente lopez'
      WHEN zona = 'bsas-gba-norte/pilar' THEN 'pilar'
      WHEN zona = 'bsas-gba-norte/tigre' THEN 'tigre'
      WHEN zona = 'bsas-gba-norte/san-isidro' THEN 'san isidro'
      WHEN zona = 'bsas-gba-norte/general-san-martin' THEN 'general san martin'
      WHEN zona = 'bsas-gba-norte/san-fernando' THEN 'san fernando'
      WHEN zona = 'bsas-gba-norte/san-miguel' THEN 'san miguel'
      WHEN zona = 'bsas-gba-norte/malvinas-argentinas' THEN 'malvinas argentinas'
      WHEN zona = 'bsas-gba-norte/escobar' THEN 'escobar'
      -- GBA Oeste
      WHEN zona = 'bsas-gba-oeste/caseros' THEN 'tres de febrero'
      WHEN zona = 'bsas-gba-oeste/castelar' THEN 'moron'
      WHEN zona = 'bsas-gba-oeste/general-rodriguez' THEN 'general rodriguez'
      WHEN zona = 'bsas-gba-oeste/hurlingham' THEN 'hurlingham'
      WHEN zona = 'bsas-gba-oeste/ituzaingo' THEN 'ituzaingo'
      WHEN zona = 'bsas-gba-oeste/la-matanza' THEN 'la matanza'
      WHEN zona = 'bsas-gba-oeste/merlo' THEN 'merlo'
      WHEN zona = 'bsas-gba-oeste/moreno' THEN 'moreno'
      WHEN zona = 'bsas-gba-oeste/moron' THEN 'moron'
      WHEN zona = 'bsas-gba-oeste/tres-de-febrero' THEN 'tres de febrero'
      -- GBA Sur
      WHEN zona = 'bsas-gba-sur/almirante-brown' THEN 'almirante brown'
      WHEN zona = 'bsas-gba-sur/avellaneda' THEN 'avellaneda'
      WHEN zona = 'bsas-gba-sur/berazategui' THEN 'berazategui'
      WHEN zona = 'bsas-gba-sur/esteban-echeverria' THEN 'esteban echeverria'
      WHEN zona = 'bsas-gba-sur/ezeiza' THEN 'ezeiza'
      WHEN zona = 'bsas-gba-sur/florencio-varela' THEN 'florencio varela'
      WHEN zona = 'bsas-gba-sur/lanus' THEN 'lanus'
      WHEN zona = 'bsas-gba-sur/la-plata' THEN 'la plata'
      WHEN zona = 'bsas-gba-sur/lomas-de-zamora' THEN 'lomas de zamora'
      WHEN zona = 'bsas-gba-sur/quilmes' THEN 'quilmes'
      -- Argenprop specific mappings
      WHEN zona RLIKE 'vicente-lopez' THEN 'vicente lopez'
      WHEN zona RLIKE 'pilar' THEN 'pilar'
      WHEN zona RLIKE 'general-san-martin' THEN 'general san martin'
      WHEN zona RLIKE 'tigre' THEN 'tigre'
      WHEN zona RLIKE 'san-fernando' THEN 'san fernando'
      WHEN zona RLIKE 'san-miguel' THEN 'san miguel'
      WHEN zona RLIKE 'malvinas-argentinas' THEN 'malvinas argentinas'
      WHEN zona RLIKE 'escobar' THEN 'escobar'
      WHEN zona RLIKE 'caseros' THEN 'tres de febrero'
      WHEN zona RLIKE 'castelar' THEN 'moron'
      WHEN zona RLIKE 'general-rodriguez' THEN 'general rodriguez'
      WHEN zona RLIKE 'hurlingham' THEN 'hurlingham'
      WHEN zona RLIKE 'ituzaingo' THEN 'ituzaingo'
      WHEN zona RLIKE 'la-matanza' THEN 'la matanza'
      WHEN zona RLIKE 'merlo' THEN 'merlo'
      WHEN zona RLIKE 'moreno' THEN 'moreno'
      WHEN zona RLIKE 'moron' THEN 'moron'
      WHEN zona RLIKE 'tres-de-febrero' THEN 'tres de febrero'
      WHEN zona RLIKE 'almirante-brown' THEN 'almirante brown'
      WHEN zona RLIKE 'avellaneda' THEN 'avellaneda'
      WHEN zona RLIKE 'berazategui' THEN 'berazategui'
      WHEN zona RLIKE 'berisso' THEN 'berisso'
      WHEN zona RLIKE 'ensenada' THEN 'ensenada'
      WHEN zona RLIKE 'esteban-echeverria' THEN 'esteban echeverria'
      WHEN zona RLIKE 'ezeiza' THEN 'ezeiza'
      WHEN zona RLIKE 'florencio-varela' THEN 'florencio varela'
      WHEN zona RLIKE 'lanus' THEN 'lanus'
      WHEN zona RLIKE 'la-plata' THEN 'la plata'
      WHEN zona RLIKE 'lomas-de-zamora' THEN 'lomas de zamora'
      WHEN zona RLIKE 'presidente-peron' THEN 'presidente peron'
      WHEN zona RLIKE 'quilmes' THEN 'quilmes'
      WHEN zona RLIKE 'san-vicente' THEN 'san vicente'
      -- ZonaProp specific mappings
      WHEN zona = 'san-isidro' THEN 'san isidro'
      -- LiderProp specific mappings
      WHEN zona = 'gba-zona-norte--vicente-lopez' THEN 'vicente lopez'
      WHEN zona = 'gba-zona-norte--pilar' THEN 'pilar'
      WHEN zona = 'gba-zona-norte--general-san-martin' THEN 'general san martin'
      WHEN zona = 'gba-zona-norte--san-isidro' THEN 'san isidro'
      WHEN zona = 'gba-zona-norte--tigre' THEN 'tigre'
      WHEN zona = 'gba-zona-norte--san-fernando' THEN 'san fernando'
      WHEN zona = 'gba-zona-norte--san-miguel' THEN 'san miguel'
      WHEN zona = 'gba-zona-norte--malvinas-argentinas' THEN 'malvinas argentinas'
      WHEN zona = 'gba-zona-norte--escobar' THEN 'escobar'
      WHEN zona = 'gba-zona-oeste--canuelas' THEN 'canuelas'
      WHEN zona = 'gba-zona-oeste--general-rodriguez' THEN 'general rodriguez'
      WHEN zona = 'gba-zona-oeste--hurlingham' THEN 'hurlingham'
      WHEN zona = 'gba-zona-oeste--ituzaingo' THEN 'ituzaingo'
      WHEN zona = 'gba-zona-oeste--la-matanza' THEN 'la matanza'
      WHEN zona = 'gba-zona-oeste--lujan' THEN 'lujan'
      WHEN zona = 'gba-zona-oeste--marcos-paz' THEN 'marcos paz'
      WHEN zona = 'gba-zona-oeste--merlo' THEN 'merlo'
      WHEN zona = 'gba-zona-oeste--moreno' THEN 'moreno'
      WHEN zona = 'gba-zona-oeste--moron' THEN 'moron'
      WHEN zona = 'gba-zona-oeste--tres-de-febrero' THEN 'tres de febrero'
      WHEN zona = 'gba-zona-sur--almirante-brown' THEN 'almirante brown'
      WHEN zona = 'gba-zona-sur--avellaneda' THEN 'avellaneda'
      WHEN zona = 'gba-zona-sur--berazategui' THEN 'berazategui'
      WHEN zona = 'gba-zona-sur--ensenada' THEN 'ensenada'
      WHEN zona = 'gba-zona-sur--san-vicente' THEN 'san vicente'
      WHEN zona = 'gba-zona-sur--esteban-echeverria' THEN 'esteban echeverria'
      WHEN zona = 'gba-zona-sur--ezeiza' THEN 'ezeiza'
      WHEN zona = 'gba-zona-sur--florencio-varela' THEN 'florencio varela'
      WHEN zona = 'gba-zona-sur--lanus' THEN 'lanus'
      WHEN zona = 'gba-zona-sur--la-plata' THEN 'la plata'
      WHEN zona = 'gba-zona-sur--lomas-de-zamora' THEN 'lomas de zamora'
      WHEN zona = 'gba-zona-sur--presidente-peron' THEN 'presidente peron'
      WHEN zona = 'gba-zona-sur--quilmes' THEN 'quilmes'
      ELSE 'no especifica'
    END AS partido,
    CASE
      -- Capital Federal
      WHEN zona IS NULL THEN 'no especifica'
      WHEN zona = 'capital-federal' THEN 'capital federal'
      -- GBA Norte
      WHEN zona LIKE 'bsas-gba-norte/%' THEN 'gba zona norte'
      WHEN zona IN ('vicente-lopez', 'pilar', 'tigre', 'san-isidro', 'general-san-martin',
                    'san-fernando', 'san-miguel', 'jose-c-paz', 'malvinas-argentinas', 'escobar') THEN 'gba zona norte'
      -- GBA Oeste
      WHEN zona LIKE 'bsas-gba-oeste/%' THEN 'gba zona oeste'
      WHEN zona IN ('caseros', 'castelar', 'general-rodriguez', 'hurlingham', 'ituzaingo',
                    'la-matanza', 'merlo', 'moreno', 'moron', 'tres-de-febrero') THEN 'gba zona oeste'
      -- GBA Sur
      WHEN zona LIKE 'bsas-gba-sur/%' THEN 'gba zona sur'
      WHEN zona IN ('almirante-brown', 'avellaneda', 'berazategui', 'berisso', 'ensenada',
                    'esteban-echeverria', 'ezeiza', 'florencio-varela', 'lanus', 'la-plata',
                    'lomas-de-zamora', 'presidente-peron', 'quilmes', 'san-vicente') THEN 'gba zona sur'
      -- LiderProp specific mappings
      WHEN zona LIKE 'gba-zona-norte--%' THEN 'gba zona norte'
      WHEN zona LIKE 'gba-zona-oeste--%' THEN 'gba zona oeste'
      WHEN zona LIKE 'gba-zona-sur--%' THEN 'gba zona sur'
      ELSE zona
    END AS region,
    tipo_operacion,
    precio,
    moneda,
    expensas,
    ambientes,
    m2_totales,
    m2_cubiertos,
    antiguedad,
    cochera,
    orientacion,
    estado,
    url,
    fecha_publicacion,
    ROUND(precio / m2_totales,2) AS precio_por_m2 
  FROM datos_deduplicados
  WHERE
    rn = 1
)
SELECT  
  partido,
  region,
  tipo_operacion,
  precio,
  moneda,
  expensas,
  ambientes,
  m2_totales,
  m2_cubiertos,
  antiguedad,
  cochera,
  orientacion,
  estado,
  url,
  fecha_publicacion,
  precio_por_m2,
  'bootcamp_de_valentin.bronze.propiedades_bronze' AS _source_table,
  current_timestamp() AS _processing_timestamp
FROM datos_finales

In [0]:
-- Ejercicio 3.3
-- Validacion de datos
SELECT
  COUNT(*)
FROM bootcamp_de_valentin.silver.propiedades_silver

UNION ALL

SELECT
  COUNT(*)
FROM bootcamp_de_valentin.bronze.propiedades_bronze

In [0]:
SELECT
  *
FROM bootcamp_de_valentin.silver.propiedades_silver
LIMIT 10

In [0]:
SELECT
  COUNT(*) total_registros,
  COUNT( CASE WHEN precio <= 0 OR precio IS NULL THEN 1 END) precios_invalidos,
  COUNT( CASE WHEN m2_totales <= 0 OR m2_totales IS NULL THEN 1 END) m2_totales_invalidos,
  COUNT( CASE WHEN m2_cubiertos < 0  THEN 1 END) m2_cubiertos_invalidos,
  COUNT( CASE WHEN antiguedad = 999 THEN 1 END) antiguedad_desconocida, 
  COUNT( CASE WHEN partido IS NULL THEN 1 END) partidos_invalidos,
  COUNT( CASE WHEN region IS NULL THEN 1 END) regiones_invalidos
FROm bootcamp_de_valentin.silver.propiedades_silver
/*
  DESCUBRIMIENTO: la mayoria de los registros (99.99%) tiene antiguedad incierta
*/


# 4. Analisis Silver - Agregaciones

In [0]:
-- Ejercicio 4.1
-- Metricas por partido
SELECT
  partido,
  tipo_operacion,
  moneda,
  count(*) total_registros,
  round( AVG(precio),2) precio_promedio,
  MIN(precio) precio_min,
  MAX(precio) precio_max,
  ROUND( AVG(precio_por_m2),2 ) precio_m2_promedio
FROM bootcamp_de_valentin.silver.propiedades_silver
GROUP BY partido, tipo_operacion, moneda
HAVING COUNT(*) >= 5
ORDER BY partido, tipo_operacion, moneda


In [0]:
-- Ejercicio 4.2
-- Ranking de partidos por precio
WITH
precio_promedio(
  SELECT
    partido,
    tipo_operacion,
    moneda,
    round( AVG(precio),2) precio_promedio
  FROM bootcamp_de_valentin.silver.propiedades_silver
  GROUP BY partido, tipo_operacion, moneda
)
SELECT
  *,
  row_number() OVER(partition by tipo_operacion, moneda ORDER BY precio_promedio DESC) Ranking
FROM precio_promedio

In [0]:
-- Ejercicio 4.3
-- Distribucion por ambientes
SELECT
  ambientes,
  tipo_operacion,
  moneda,
  count(*) cantidad_registros,
  SUM(count(*)) OVER(PARTITION BY tipo_operacion, moneda ) total_registros,
  ROUND( (COUNT(*) / SUM(count(*)) OVER(PARTITION BY tipo_operacion, moneda ) ) *100,2) pct_total
FROM bootcamp_de_valentin.silver.propiedades_silver
GROUP BY
  ambientes,
  tipo_operacion,
  moneda
ORDER BY
ambientes,
  tipo_operacion,
  moneda,
  pct_total DESC

# 5. Verificacion Pipeline

In [0]:
-- Ejercicio 5.1
-- Listado de objetos por capa
SELECT
  *
FROM bootcamp_de_valentin.information_schema.tables
WHERE
  table_schema IN ('bronze', 'silver', 'gold')
ORDER BY
  table_schema

In [0]:
-- Ejercicio 5.2
-- Conteo registros por capa
SELECT
  'Bronze' as capa,
  COUNT(*) as total_registros
FROM bootcamp_de_valentin.bronze.propiedades_bronze

UNION ALL

SELECT
  'Silver' as capa,
  COUNT(*) as total_registros
FROM bootcamp_de_valentin.silver.propiedades_silver


In [0]:
-- Ejercicio 5.3
-- Registros filtrados por el pipeline
SELECT
  *
FROM bronze_eda
WHERE
  precio <= 0
  OR precio is NULL
  OR m2_totales <= 0
  OR m2_totales is NULL
  OR tipo_de_operacion is null